In [19]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import AgglomerativeClustering

In [20]:
CSV_PATH = "raw_wholesale_customers.csv"
FEATURES = ["Fresh", "Milk", "Grocery",
            "Frozen", "Detergents_Paper", "Delicassen"]
RANDOM_STATE = 42
K = 5

## 1 : Load Data 

In [21]:
df = pd.read_csv(CSV_PATH)
print("\n=== INITIAL SNAPSHOT ===")
print(df.head())
print(df.info())


=== INITIAL SNAPSHOT ===
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185
<class 'pandas.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen            440 non-null    int64
 6   Detergents_Paper  440 non-null    int64
 7

In [22]:
X =df[FEATURES].copy()

## 2) Select features + IQR cap
## L3: clip extreme spend values before scaling — same idea as house/loan pipelines

In [23]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper


low_fresh,  high_fresh = iqr_fun(X["Fresh"])
low_milk,    high_milk = iqr_fun(X["Milk"])
low_grocery, high_grocery = iqr_fun(X["Grocery"])
low_frozen,  high_frozen = iqr_fun(X["Frozen"])
low_det,     high_det = iqr_fun(X["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(X["Delicassen"])

X["Fresh"] = X["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)

df[FEATURES] = X

print("\n=== IQR CAP SUMMARY FOR EACH FEATURE ===")
print(
    f"Fresh             low={low_fresh:.2f}  high={high_fresh:.2f}  |  after cap min={X['Fresh'].min():.2f}  max={X['Fresh'].max():.2f}")
print(
    f"Milk              low={low_milk:.2f}  high={high_milk:.2f}  |  after cap min={X['Milk'].min():.2f}  max={X['Milk'].max():.2f}")
print(
    f"Grocery           low={low_grocery:.2f}  high={high_grocery:.2f}  |  after cap min={X['Grocery'].min():.2f}  max={X['Grocery'].max():.2f}")
print(
    f"Frozen            low={low_frozen:.2f}  high={high_frozen:.2f}  |  after cap min={X['Frozen'].min():.2f}  max={X['Frozen'].max():.2f}")
print(
    f"Detergents_Paper  low={low_det:.2f}  high={high_det:.2f}  |  after cap min={X['Detergents_Paper'].min():.2f}  max={X['Detergents_Paper'].max():.2f}")
print(
    f"Delicassen        low={low_deli:.2f}  high={high_deli:.2f}  |  after cap min={X['Delicassen'].min():.2f}  max={X['Delicassen'].max():.2f}")

print("\n=== FEATURES HEAD (after IQR cap) ===")
print(X.head())


=== IQR CAP SUMMARY FOR EACH FEATURE ===
Fresh             low=-17581.25  high=37642.75  |  after cap min=3.00  max=37642.75
Milk              low=-6952.88  high=15676.12  |  after cap min=55.00  max=15676.12
Grocery           low=-10601.12  high=23409.88  |  after cap min=3.00  max=23409.88
Frozen            low=-3475.75  high=7772.25  |  after cap min=25.00  max=7772.25
Detergents_Paper  low=-5241.12  high=9419.88  |  after cap min=3.00  max=9419.88
Delicassen        low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25

=== FEATURES HEAD (after IQR cap) ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


## 3: Scale features

In [24]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nScaled shape:", X_scaled.shape)


Scaled shape: (440, 6)


## 4: Elbow method (print SSE)

In [25]:
print("\n ELBOW METHOD (SSE per k)")
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    km.fit(X_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")


 ELBOW METHOD (SSE per k)
k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


## 5 : Fit K-Means with chosen k


In [26]:

kmeans = KMeans(n_clusters=K, n_init="auto", random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = labels.astype(int)
print("\n=== SAMPLE WITH CLUSTERS ===")
print(df.head())


=== SAMPLE WITH CLUSTERS ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        0  
1     1776.00        0  
2     3938.25        0  
3     1788.00        3  
4     3938.25        3  


## 6: Evaluate clustering

In [27]:
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
print("\n=== METRICS ===")
print("the K is : ", K)
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")


=== METRICS ===
the K is :  5
Silhouette Score : 0.283 (closer to +1 is better)
Davies–Bouldin   : 1.270 (lower is better)


## 7: Cluster centers (back to original units)

In [28]:
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"

print("\n=== CLUSTER CENTERS (Original Units) ===")
print(centers_df.round(2))


=== CLUSTER CENTERS (Original Units) ===
            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         9202.67   6833.30   9104.12  1326.16           3280.12     1871.76
1         8376.23   2150.65   3160.63  1646.33            779.25      674.02
2        17461.54  13805.60  17524.12  4120.57           5460.56     3583.64
3        22346.70   3409.14   3969.33  5819.60            583.07     1566.95
4         4916.98  10768.85  18350.13  1212.37           7780.02      981.37


## 8: Second clustering algorithm — Agglomerative Clustering

I chose **Agglomerative (Hierarchical) Clustering** because it doesn't assume
clusters are round/evenly-sized like K-Means does, and it lets us inspect the
merge structure (dendrogram) to sanity-check how many clusters make sense —
useful for exploratory customer segmentation where we don't know the "true"
number of groups in advance.

Source: scikit-learn docs — https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html

In [29]:
agglo = AgglomerativeClustering(n_clusters=K)
agglo_labels = agglo.fit_predict(X_scaled)

df["Cluster_Agglo"] = agglo_labels.astype(int)
print("\n=== SAMPLE WITH AGGLOMERATIVE CLUSTERS ===")
print(df.head())


=== SAMPLE WITH AGGLOMERATIVE CLUSTERS ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  Cluster_Agglo  
0     1338.00        0              4  
1     1776.00        0              4  
2     3938.25        0              0  
3     1788.00        3              3  
4     3938.25        3              0  


## 9 : Evaluating Agglo

In [30]:
sil_agglo = silhouette_score(X_scaled, agglo_labels)
dbi_agglo = davies_bouldin_score(X_scaled, agglo_labels)

print("\n=== AGGLOMERATIVE METRICS ===")
print("the K is : ", K)
print(f"Silhouette Score : {sil_agglo:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi_agglo:.3f} (lower is better)")



=== AGGLOMERATIVE METRICS ===
the K is :  5
Silhouette Score : 0.218 (closer to +1 is better)
Davies–Bouldin   : 1.325 (lower is better)


## 10: Compare K-Means vs Agglomerative Clustering


In [31]:
print("\n=== METHOD COMPARISON ===")
print(f"K-Means        → Silhouette: {sil:.3f}   Davies-Bouldin: {dbi:.3f}")
print(f"Agglomerative  → Silhouette: {sil_agglo:.3f}   Davies-Bouldin: {dbi_agglo:.3f}")

if sil > sil_agglo:
    print("\nK-Means produced better-separated clusters (higher Silhouette Score).")
elif sil_agglo > sil:
    print("\nAgglomerative Clustering produced better-separated clusters (higher Silhouette Score).")
else:
    print("\nBoth methods produced equally separated clusters.")



=== METHOD COMPARISON ===
K-Means        → Silhouette: 0.283   Davies-Bouldin: 1.270
Agglomerative  → Silhouette: 0.218   Davies-Bouldin: 1.325

K-Means produced better-separated clusters (higher Silhouette Score).


## 11 : Sanity Check (10 Cleints)

In [32]:
sample_idx = [0, 1, 2,3,4,5,6,7,8,9]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster", "Cluster_Agglo"]]
print("\n=== SANITY CHECK (10 Clients, Both Methods) ===")
print(sanity)


=== SANITY CHECK (10 Clients, Both Methods) ===
   Channel  Region    Fresh     Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0   9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0   9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0   8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0   1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0   5410.0   7198.0  3915.0            1777.0   
5        2       3   9413.0   8259.0   5126.0   666.0            1795.0   
6        2       3  12126.0   3199.0   6975.0   480.0            3140.0   
7        2       3   7579.0   4956.0   9426.0  1669.0            3321.0   
8        1       3   5963.0   3648.0   6192.0   425.0            1716.0   
9        2       3   6006.0  11093.0  18881.0  1159.0            7425.0   

   Delicassen  Cluster  Cluster_Agglo  
0     1338.00        0              4  
1     1776.00        0              4  
2    

## 12 : Save

In [33]:
OUT_PATH = "segmented_wholesale_customers.csv"
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved to {OUT_PATH}")



Saved to segmented_wholesale_customers.csv
